In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/dataset.zip" -d "/content/dataset/"


Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/dataset/Spoiled/S_Banana/101.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/102.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/103.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/104.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/105.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/106.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/107.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/108.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/109.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/11.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/110.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/111.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/112.jpg  
  inflating: /content/dataset/dataset/Spoiled/S_Banana/113.jpg  
  inflating: /content/dataset/dataset/Sp

ا**لطريقة الاولى      غير معتمدة**




In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
import os


In [ ]:
dataset_dir = "/content/dataset/dataset"   # عدّليه حسب مكان ملفاتك
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)


Found 8000 files belonging to 2 classes.
Using 6400 files for training.
Found 8000 files belonging to 2 classes.
Using 1600 files for validation.


In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


In [ ]:
# vit_url = "https://tfhub.dev/sayakpaul/vit_b16_fe/1"


# vit_layer = hub.KerasLayer(vit_url, trainable=False)
import tensorflow_hub as hub

vit_layer = hub.KerasLayer("https://tfhub.dev/google/vit_b16/clam/1",trainable=True)



OSError: https://tfhub.dev/google/vit_b16/clam/1 does not appear to be a valid module.

In [ ]:
import tensorflow_hub as hub
import tensorflow as tf

vit_url = "https://tfhub.dev/sayakpaul/vit_b16_fe/1"
vit_layer = hub.KerasLayer(vit_url, trainable=False)

# نحوله لـ model لكي نقدر نستعرض الطبقات
model = tf.keras.Sequential([vit_layer])

model.summary()


ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x79e991bc2930> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)

In [ ]:

import tensorflow as tf
import tensorflow_hub as hub
vit_url = "https://tfhub.dev/sayakpaul/vit_b16_fe/1"

# vit_url = "https://tfhub.dev/sayakpaul/vit_tiny16_fe/1"
# vit_url = "https://tfhub.dev/google/vit_tiny_patch16_224/1"


# إزالة input_shape لتجنب التعارض
vit_layer = hub.KerasLayer(vit_url, trainable=False)
# عدد الأصناف من الداتا
num_classes = train_ds.element_spec[1].shape[1]
# بناء النموذج باستخدام Functional API مع إصلاح لـ hub.KerasLayer
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1./255)(x)
# تغليف vit_layer بـ Lambda لتجنب الخطأ مع KerasTensor
x = tf.keras.layers.Lambda(lambda x: vit_layer(x))(x)
x = tf.keras.layers.Dropout(0.1)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)


model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,538 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,538 (6.01 KB)

 Trainable params: 1,538 (6.01 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 117s 505ms/step - accuracy: 0.5674 - loss: 1.1832 - val_accuracy: 0.7594 - val_loss: 0.5360
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 472ms/step - accuracy: 0.7970 - loss: 0.4688 - val_accuracy: 0.8975 - val_loss: 0.2488
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 470ms/step - accuracy: 0.8870 - loss: 0.2760 - val_accuracy: 0.9425 - val_loss: 0.1551
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 469ms/step - accuracy: 0.9299 - loss: 0.1806 - val_accuracy: 0.9606 - val_loss: 0.1124
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 470ms/step - accuracy: 0.9491 - loss: 0.1382 - val_accuracy: 0.9725 - val_loss: 0.0881


In [ ]:
vit_layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine_tune = model.fit(train_ds, validation_data=val_ds, epochs=6)


Epoch 1/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 100s 485ms/step - accuracy: 0.9539 - loss: 0.1136 - val_accuracy: 0.9737 - val_loss: 0.0842
Epoch 2/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 472ms/step - accuracy: 0.9569 - loss: 0.1102 - val_accuracy: 0.9744 - val_loss: 0.0813
Epoch 3/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 470ms/step - accuracy: 0.9621 - loss: 0.1041 - val_accuracy: 0.9762 - val_loss: 0.0780
Epoch 4/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 473ms/step - accuracy: 0.9671 - loss: 0.1005 - val_accuracy: 0.9762 - val_loss: 0.0754
Epoch 5/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 472ms/step - accuracy: 0.9660 - loss: 0.0942 - val_accuracy: 0.9769 - val_loss: 0.0729
Epoch 6/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 471ms/step - accuracy: 0.9679 - loss: 0.0988 - val_accuracy: 0.9781 - val_loss: 0.0703


In [ ]:
import numpy as np

trainable_count = sum([1 for l in model.layers if l.trainable])
total_count = len(model.layers)
print("Trainable layers:", trainable_count, "/", total_count)


trainable_params = np.sum([np.prod(v.shape) for v in model.trainable_weights])
non_trainable_params = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])
print("Trainable params:", trainable_params, "Non-trainable params:", non_trainable_params)


Trainable layers: 6 / 6
Trainable params: 1538 Non-trainable params: 0.0


In [ ]:
loss, acc = model.evaluate(val_ds)
print("Accuracy:", acc)


50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 415ms/step - accuracy: 0.9744 - loss: 0.0728
Accuracy: 0.9781249761581421


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np  # للتحقق من المعلمات

# تحضير البيانات (كما هو في كودك)
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# تعريف data_augmentation (كما هو في كودك)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# رابط الـ ViT Feature Extractor
vit_url = "https://tfhub.dev/sayakpaul/vit_b16_fe/1"

# عدد الأصناف من الداتا
num_classes = train_ds.element_spec[1].shape[1]

# دالة لتغليف vit_layer باستخدام tf.py_function (للسماح بتحديث trainable)
def vit_layer_func(x, trainable=True):
    vit_layer = hub.KerasLayer(vit_url, trainable=trainable)
    return vit_layer(x)

# بناء النموذج الأول (مع تجميد ViT)
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1./255)(x)
x = tf.keras.layers.Lambda(lambda x: vit_layer_func(x, trainable=False))(x)  # تجميد ViT
x = tf.keras.layers.Dropout(0.1)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

# تجميع وتدريب أولي (مع تجميد ViT)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()  # للتحقق من الطبقات
history = model.fit(train_ds, validation_data=val_ds, epochs=5)



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,538 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,538 (6.01 KB)

 Trainable params: 1,538 (6.01 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 134s 527ms/step - accuracy: 0.5411 - loss: 1.1823 - val_accuracy: 0.8069 - val_loss: 0.4350
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 95s 474ms/step - accuracy: 0.8129 - loss: 0.4088 - val_accuracy: 0.9075 - val_loss: 0.2338
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 472ms/step - accuracy: 0.9061 - loss: 0.2257 - val_accuracy: 0.9525 - val_loss: 0.1436
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 471ms/step - accuracy: 0.9455 - loss: 0.1570 - val_accuracy: 0.9650 - val_loss: 0.1107
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 472ms/step - accuracy: 0.9577 - loss: 0.1192 - val_accuracy: 0.9719 - val_loss: 0.0907


لما فكينا كل الطبقات سوا بهي الطريقة استهلك كل الرام لانو كثير كبير هذا ال VIT ف عطانا Your session crashed after using all available RAM

In [ ]:
inputs_ft = tf.keras.Input(shape=(224, 224, 3))
x_ft = data_augmentation(inputs_ft)
x_ft = tf.keras.layers.Rescaling(1./255)(x_ft)
x_ft = tf.keras.layers.Lambda(lambda x: vit_layer_func(x, trainable=True))(x_ft)  # فك التجميد
x_ft = tf.keras.layers.Dropout(0.1)(x_ft)
outputs_ft = tf.keras.layers.Dense(num_classes, activation="softmax")(x_ft)
model_ft = tf.keras.Model(inputs_ft, outputs_ft)  # نموذج جديد للـ fine-tune

# تجميع وتدريب fine-tune (مع معدل تعلم أقل)
model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model_ft.summary()  # تحقق من أن المعلمات الآن ملايين (trainable params يجب أن يكون كبيراً جداً)
history_fine_tune = model_ft.fit(train_ds, validation_data=val_ds, epochs=6)


NameError: name 'tf' is not defined

In [ ]:


# تحقق من الطبقات القابلة للتدريب بعد الإصلاح
trainable_count = sum([1 for l in model_ft.layers if l.trainable])
total_count = len(model_ft.layers)
print("Trainable layers:", trainable_count, "/", total_count)

trainable_params = np.sum([np.prod(v.shape) for v in model_ft.trainable_weights])
non_trainable_params = np.sum([np.prod(v.shape) for v in model_ft.non_trainable_weights])
print("Trainable params:", trainable_params, "Non-trainable params:", non_trainable_params)

**الطريقة الثانية  معتمدة**



 نتائجها افضل  


In [ ]:
!pip install transformers datasets accelerate timm


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import ViTFeatureExtractor, ViTForImageClassification
import os
import random
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
DATA_DIR = "/content/dataset/dataset"

model_name = "google/vit-base-patch16-224-in21k"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
])

val_transform = transforms.Resize((224,224))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, labels, feature_extractor, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.feature_extractor = feature_extractor
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        encoding = self.feature_extractor(images=image, return_tensors="pt")
        pixel_values = encoding['pixel_values'].squeeze()
        return pixel_values, label


In [ ]:
def get_all_image_paths_labels(root_dir):
    image_paths = []
    labels = []
    class_names = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

    for idx, main_class in enumerate(class_names):
        main_class_path = os.path.join(root_dir, main_class)

        for subfolder in os.listdir(main_class_path):
            subfolder_path = os.path.join(main_class_path, subfolder)
            if os.path.isdir(subfolder_path):
                for img_name in os.listdir(subfolder_path):
                    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
                        image_paths.append(os.path.join(subfolder_path, img_name))
                        labels.append(idx)

    return image_paths, labels, class_names


all_image_paths, all_labels, class_names = get_all_image_paths_labels(DATA_DIR)

print("Classes:", class_names)
print("Total images:", len(all_image_paths))

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_image_paths, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")


Classes: ['Frish', 'Spoiled']
Total images: 10000
Train: 7000  Val: 1500  Test: 1500


In [ ]:
train_dataset = CustomImageDataset(train_paths, train_labels, feature_extractor, transform=train_transform)
val_dataset   = CustomImageDataset(val_paths, val_labels, feature_extractor, transform=val_transform)
test_dataset  = CustomImageDataset(test_paths, test_labels, feature_extractor, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
num_classes = len(class_names)

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)
model.to(device)


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
for param in model.vit.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print("Trainable:", trainable)
print("Frozen:", frozen)


Trainable: 1538
Frozen: 85798656


In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

criterion = torch.nn.CrossEntropyLoss()


In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, leave=False)

    for pixel_values, labels in loop:
        pixel_values, labels = pixel_values.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(pixel_values=pixel_values).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            logits = model(pixel_values=pixel_values).logits
            preds = logits.argmax(1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return correct / total


In [ ]:
epochs = 5
for epoch in range(epochs):
    loss = train_one_epoch()
    acc = evaluate()
    print(f"[Frozen] Epoch {epoch+1}/{epochs} — Loss: {loss:.4f} — Val Acc: {acc:.4f}")


  0%|          | 0/438 [00:00<?, ?it/s]

[Frozen] Epoch 1/5 — Loss: 0.6149 — Val Acc: 0.9040


  0%|          | 0/438 [00:00<?, ?it/s]

[Frozen] Epoch 2/5 — Loss: 0.5027 — Val Acc: 0.9340


  0%|          | 0/438 [00:00<?, ?it/s]

[Frozen] Epoch 3/5 — Loss: 0.4244 — Val Acc: 0.9447


  0%|          | 0/438 [00:00<?, ?it/s]

[Frozen] Epoch 4/5 — Loss: 0.3664 — Val Acc: 0.9500


  0%|          | 0/438 [00:00<?, ?it/s]

[Frozen] Epoch 5/5 — Loss: 0.3219 — Val Acc: 0.9547


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

test_acc = correct / total
print("Test Accuracy:", test_acc)


Test Accuracy: 0.9658333333333333


In [ ]:
for param in model.vit.encoder.layer[-1].parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)


In [ ]:
epochs = 3
for epoch in range(epochs):
    loss = train_one_epoch()
    acc = evaluate()
    print(f"[Unfrozen Last Block] Epoch {epoch+1}/{epochs} — Loss: {loss:.4f} — Val Acc: {acc:.4f}")


  0%|          | 0/438 [00:00<?, ?it/s]

[Unfrozen Last Block] Epoch 1/3 — Loss: 0.1923 — Val Acc: 0.9847


  0%|          | 0/438 [00:00<?, ?it/s]

[Unfrozen Last Block] Epoch 2/3 — Loss: 0.0716 — Val Acc: 0.9947


  0%|          | 0/438 [00:00<?, ?it/s]

[Unfrozen Last Block] Epoch 3/3 — Loss: 0.0289 — Val Acc: 0.9960


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

test_acc = correct / total
print("Test Accuracy:", test_acc)


Test Accuracy: 0.9975


In [ ]:

for param in model.vit.parameters():
    param.requires_grad = True


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-6
)

print("All layers unfrozen ✓")


All layers unfrozen ✓


In [ ]:
epochs = 3
for epoch in range(epochs):
    loss = train_one_epoch()
    acc = evaluate()
    print(f"[Full Unfreeze] Epoch {epoch+1}/{epochs} — Loss: {loss:.4f} — Val Acc: {acc:.4f}")


  0%|          | 0/438 [00:00<?, ?it/s]

[Full Unfreeze] Epoch 1/3 — Loss: 0.0069 — Val Acc: 1.0000


  0%|          | 0/438 [00:00<?, ?it/s]

[Full Unfreeze] Epoch 2/3 — Loss: 0.0017 — Val Acc: 1.0000


  0%|          | 0/438 [00:00<?, ?it/s]

[Full Unfreeze] Epoch 3/3 — Loss: 0.0012 — Val Acc: 1.0000


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

test_acc = correct / total
print("Test Accuracy:", test_acc)


Test Accuracy: 1.0


In [ ]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score


all_preds = []
all_labels_real = []

model.eval()
with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels_real.extend(labels.cpu().numpy())


f1 = f1_score(all_labels_real, all_preds, average='weighted')
precision = precision_score(all_labels_real, all_preds, average='weighted')
recall = recall_score(all_labels_real, all_preds, average='weighted')

print("------ Evaluation Metrics ------")
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


print("\nClassification Report:")
print(classification_report(all_labels_real, all_preds, target_names=class_names))


------ Evaluation Metrics ------
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000

Classification Report:
              precision    recall  f1-score   support

       Frish       1.00      1.00      1.00       600
     Spoiled       1.00      1.00      1.00       600

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200



In [ ]:
save_path = "/content/drive/MyDrive/vit_fruit_model.pth"
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)


Model saved to: /content/drive/MyDrive/vit_fruit_model.pth


In [ ]:
import numpy as np

def predict_image(img_path):
    image = Image.open(img_path).convert("RGB")
    image = val_transform(image)

    encoding = feature_extractor(images=image, return_tensors="pt")
    pixel_values = encoding["pixel_values"].to(device)

    model.eval()
    with torch.no_grad():
        logits = model(pixel_values=pixel_values).logits
        pred = logits.argmax(1).item()

    return class_names[pred]

test_img = test_paths[0]
print("Image:", test_img)
print("Prediction:", predict_image(test_img))


Image: /content/dataset/dataset/Frish/F_Lemon/806.jpg
Prediction: Frish
